# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. My lane on the ML loop

**Lane:** Refresh / Content Opportunity Scoring — *which pages should editors review first for refresh, expansion, protection, pruning, or monitoring?*

| ML loop stage | In this lane |
|---|---|
| **World / Reality** | Thousands of content pages across clients, each with search visibility, freshness, CTR, position, and engagement signals that shift over time. |
| **Data** | Starter slice: `data/raw/content_refresh_anonymized.csv` (~30k rows). Observable signals only — no product decision flags. |
| **Features** | Per-page 90-day impressions, sessions, age, days since update, CTR, avg position, trend direction, content type, intent, etc. |
| **Model** | Learned scorer (e.g., logistic regression or tree ensemble) that estimates decline/opportunity risk from many signals at once. |
| **Prediction / Score** | A refresh priority score (0–100) plus reason codes per page. |
| **Decision / Action** | Editors act on a **ranked review queue**: work top-K pages first instead of scanning the whole inventory. |
| **Changed world → new data** | After refreshes, future performance windows become new labels for retraining and validation. |

**Task type:** **Ranking / scoring** (primary). We output an ordered queue, not a single yes/no for the whole site.

**Action the output supports:** Editors use the ranked queue to decide **which page to open first** — refresh stale high-visibility pages, expand thin winners, protect declining page-one assets, or deprioritize low-signal pages.

**Why ML beats a fixed rule here:** Decline risk depends on many signals interacting (visibility × freshness × position × CTR × engagement). A hand-written if-statement can flag obvious cases but misses subtler combinations; the starter baseline reached only **Precision@50 = 0.24** vs **0.74** for a random forest on the same proxy label.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


In [3]:
# Ranking/Scoring is the primary ML task, with classification used to estimate the likelihood of a page declining.

# Primary output: A ranked list of pages with refresh priority scores.
# Why ranking? Editors need a prioritized list, not just a yes/no prediction, so they can focus on the highest-impact pages first.
# Why classification? The model predicts the probability of decline, which is used to generate the final ranking.
# Why not clustering? Clustering groups similar pages but does not help prioritize which pages should be reviewed first.

# Use case: Helps content and SEO teams identify which pages to refresh, improve, or monitor first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# Starter proxy target (current prediction):

# is_declining_label = 1 if trend_direction == "down" else 0

# Scored unit: An individual anonymized content page (`content_id`) for a client.
# Label source: Based on the observed 90-day trend using `trend_direction` and `trend_pct`.
# Limitation: This is a **proxy target**, indicating pages with declining performance rather than guaranteeing a successful refresh.
# Future improvement: Predict future decline or recovery using historical data instead of relying on current trends.
# Note: The model does not prove that content updates caused recovery or predict search engine algorithms.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# Primary Metric: Precision@50

# Measures the percentage of truly declining pages within the **top 50** ranked by the model.

# Why this metric? Editors review only a small set of pages, so ranking quality at the top is more important than overall accuracy.
# Starter results: Baseline score = 0.24 (~12/50 correct), Random Forest = 0.74 (~37/50 correct), showing a significant improvement.
# Supporting metrics: Average Precision and manual review of the top 20 results.
# Not optimized: Overall accuracy, as it can be misleading due to the high number of declining pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
from pathlib import Path
import pandas as pd

relative_data_path = Path("data") / "raw" / "content_refresh_anonymized.csv"

repo_root = None
for base in [Path.cwd(), *Path.cwd().parents]:
    candidate = base / relative_data_path
    if candidate.exists():
        repo_root = base
        data_path = candidate
        break

if repo_root is None:
    raise FileNotFoundError(f"Could not find {relative_data_path}")

raw = pd.read_csv(data_path)

lane = (
    raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)]
    .drop_duplicates(subset=["content_id"])
    .copy()
)

lane["is_declining_label"] = (
    lane["trend_direction"].str.lower().eq("down").astype(int)
)

unit_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

print(f"Loaded: {data_path}")
print("Unit of analysis: one row = one content page (content_id) for one client")
print(
    f"Lane slice rows: {len(lane):,} | Declining proxy rate: "
    f"{lane['is_declining_label'].mean():.1%}"
)
print()

display(lane[unit_cols].head(8))

print("\nTarget column value counts:")
print(
    lane["is_declining_label"]
    .value_counts()
    .rename({0: "Not Declining (Proxy)", 1: "Declining (Proxy)"})
)

print("\nSample high-visibility declining pages:")
high_vis_declining = (
    lane[
        (lane["is_declining_label"] == 1)
        & (lane["impressions_90d"] >= 500)
    ]
    .nlargest(5, "impressions_90d")
)

display(high_vis_declining[unit_cols])

Loaded: c:\Users\Huniza Computer\Downloads\Mustafa-FlyRank-main\Mustafa-FlyRank\data\raw\content_refresh_anonymized.csv
Unit of analysis: one row = one content page (content_id) for one client
Lane slice rows: 30,000 | Declining proxy rate: 54.2%



,content_id,client_id,impressions_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,20,0.76,10.6,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,25,0.05,20.3,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,20,0.09,36.5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,11751,78,463,22,0.49,6.2,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,14,0.13,44.0,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,20,0.03,8.5,down,-38.9,1
6,content_9a34b442b552,client_8722616204,20,1,90,20,0.00,7.0,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,1724,28,445,22,0.06,21.2,stable,0.6,0



Target column value counts:
is_declining_label
Declining (Proxy)        16262
Not Declining (Proxy)    13738
Name: count, dtype: int64

Sample high-visibility declining pages:


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,trend_direction,trend_pct,is_declining_label
6653,content_5fe46e04994d,client_4e07408562,517715,520,537,104,0.14,4.2,down,-44.8,1
26844,content_8c19996aa890,client_4e07408562,509252,571,445,20,0.15,2.5,down,-44.5,1
21819,content_4c36c775b818,client_4e07408562,463103,1114,445,20,0.41,2.3,down,-33.2,1
29879,content_1a9e894be2e2,client_19581e27de,416180,1140,482,22,0.23,4.0,down,-27.0,1
13537,content_2c2606c5d176,client_19581e27de,347399,2146,362,104,0.53,4.2,down,-36.5,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# The baseline rule relies on fixed weights for factors like **visibility, freshness, position, and depth**, which works for simple cases but cannot capture complex feature interactions.

# A machine learning model learns these patterns automatically, producing more accurate rankings. On the starter dataset, it improved Precision@50 from 0.24 to 0.74 while still providing interpretable results for editors.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.